In [ ]:
# ════════════════════════════════════════════════════════════════
# CELL 1 — INSTALL DEPENDENCIES   ▶  RUN THIS FIRST (required)
# Installs the libraries the rest of the notebook depends on,
# then restarts Python so the new packages load. Without this,
# subsequent cells will fail with import errors.
# ════════════════════════════════════════════════════════════════
%pip install "databricks-openai[memory]>=0.13.0" "databricks-sdk>=0.70.0"
dbutils.library.restartPython()

In [ ]:
# ════════════════════════════════════════════════════════════════
# CELL 2 — LIST BRANCHES & ENDPOINTS   ▶  RUN THIS (optional, for verification)
# Confirms your Lakebase project, branch, and endpoint names so you
# can copy the right values into databricks.yml. Set PROJECT_ID first.
# ════════════════════════════════════════════════════════════════
from databricks.sdk import WorkspaceClient
import json

w = WorkspaceClient()

PROJECT_ID = "<your lakebase database project ID/name>"

# List branches to find the correct branch ID
branches = w.api_client.do("GET", f"/api/2.0/postgres/projects/{PROJECT_ID}/branches")
print("Branches:")
for b in branches.get("branches", []):
    print(f"  {b['name']}")
    branch_id = b["name"].split("/branches/")[-1]

    # List endpoints for each branch
    endpoints = w.api_client.do("GET", f"/api/2.0/postgres/projects/{PROJECT_ID}/branches/{branch_id}/endpoints")
    for ep in endpoints.get("endpoints", []):
        print(f"    Endpoint: {ep['name']}")
        print(f"    Host: {ep.get('status', {}).get('host', 'N/A')}")

In [ ]:
# ════════════════════════════════════════════════════════════════
# CELL 3 — CREATE AGENT MEMORY TABLES   ✗  DO NOT RUN (default flow)
# The deployed app auto-creates these tables on first startup. Running
# this cell manually as YOUR user makes you the table owner, which
# blocks the app's service principal from writing later (you'd then
# need the GRANT statements from the setup guide). Only run this if
# you're deliberately pre-creating tables (e.g. Path B UI fallback).
# ════════════════════════════════════════════════════════════════
from databricks_openai.agents.session import AsyncDatabricksSession

# Replace with YOUR Lakebase endpoint path from Step 2
LAKEBASE_ENDPOINT = "projects/<project name>/branches/<branch name>/endpoints/primary"

async def create_agent_tables():
    session = AsyncDatabricksSession(
        session_id="__setup__",
        autoscaling_endpoint=LAKEBASE_ENDPOINT,
        schema="agent_openai_memory",
    )
    await session._ensure_tables()
    print("✓ Backend agent memory tables created (schema: agent_openai_memory)")

await create_agent_tables()

In [ ]:
# ════════════════════════════════════════════════════════════════
# CELL 4 — LIST DATABASES   ▶  RUN THIS (required for databricks.yml)
# Prints the full database path you need to paste into the `database:`
# field of the postgres resource in databricks.yml. Replace
# <project name> with your Lakebase project name before running.
# ════════════════════════════════════════════════════════════════
from databricks.sdk import WorkspaceClient
import json

w = WorkspaceClient()

# List databases for the project/branch
response = w.api_client.do(
    "GET",
    "/api/2.0/postgres/projects/<project name>/branches/production/databases"
)
print(json.dumps(response, indent=2))

In [ ]:
# ════════════════════════════════════════════════════════════════
# CELL 5 — CREATE CHAT HISTORY TABLES   ✗  DO NOT RUN (default flow)
# Left COMMENTED OUT on purpose. The Node frontend runs Drizzle
# migrations on first startup and creates these tables itself. Only
# uncomment + run if you're using the Path B (UI) deploy and hit
# "permission denied" errors — see the setup guide's troubleshooting.
# ════════════════════════════════════════════════════════════════
# from databricks_ai_bridge.lakebase import LakebaseClient

# client = LakebaseClient(
#     autoscaling_endpoint="projects/<project name>/branches/<branch name>/endpoints/primary"
# )

# client.execute("CREATE SCHEMA IF NOT EXISTS ai_chatbot;")

# client.execute('''
# CREATE TABLE IF NOT EXISTS ai_chatbot."User" (
#     id text PRIMARY KEY,
#     email varchar(64) NOT NULL
# );
# ''')
# client.execute('''
# CREATE TABLE IF NOT EXISTS ai_chatbot."Chat" (
#     id uuid PRIMARY KEY DEFAULT gen_random_uuid(),
#     "createdAt" timestamp NOT NULL,
#     title text NOT NULL,
#     "userId" text NOT NULL,
#     visibility varchar DEFAULT 'private' NOT NULL,
#     "lastContext" jsonb
# );
# ''')
# client.execute('''
# CREATE TABLE IF NOT EXISTS ai_chatbot."Message" (
#     id uuid PRIMARY KEY DEFAULT gen_random_uuid(),
#     "chatId" uuid NOT NULL REFERENCES ai_chatbot."Chat"(id),
#     role varchar NOT NULL,
#     parts json NOT NULL,
#     attachments json NOT NULL,
#     "createdAt" timestamp NOT NULL,
#     "traceId" text
# );
# ''')
# client.execute('''
# CREATE TABLE IF NOT EXISTS ai_chatbot."Vote" (
#     "chatId" uuid NOT NULL REFERENCES ai_chatbot."Chat"(id),
#     "messageId" uuid NOT NULL REFERENCES ai_chatbot."Message"(id),
#     "isUpvoted" boolean NOT NULL,
#     PRIMARY KEY("chatId", "messageId")
# );
# ''')

# print("✓ Schema ai_chatbot and all tables created.")
# client.close()

In [ ]:
# ════════════════════════════════════════════════════════════════
# CELL 6 — GRANT PERMISSIONS   ✗  DO NOT RUN (default flow)
# Left COMMENTED OUT on purpose. Only needed if YOU created the
# Lakebase tables (via cells 3 / 5 or by running the app locally
# first), since you'd then own them and the app's service principal
# can't write. If the app created its own tables (default flow),
# no grants are required.
# ════════════════════════════════════════════════════════════════
# from databricks_ai_bridge.lakebase import LakebaseClient

# client = LakebaseClient(
#     autoscaling_endpoint="projects/<project name>/branches/<branch name>/endpoints/primary"
# )

# grants = [
#     # Backend agent memory schema
#     "GRANT USAGE ON SCHEMA agent_openai_memory TO PUBLIC;",
#     "GRANT ALL ON ALL TABLES IN SCHEMA agent_openai_memory TO PUBLIC;",
#     "GRANT USAGE, SELECT, UPDATE ON ALL SEQUENCES IN SCHEMA agent_openai_memory TO PUBLIC;",
#     "ALTER DEFAULT PRIVILEGES IN SCHEMA agent_openai_memory GRANT ALL ON TABLES TO PUBLIC;",
#     "ALTER DEFAULT PRIVILEGES IN SCHEMA agent_openai_memory GRANT USAGE, SELECT, UPDATE ON SEQUENCES TO PUBLIC;",
#     # Frontend chat history schema
#     "GRANT USAGE ON SCHEMA ai_chatbot TO PUBLIC;",
#     "GRANT ALL ON ALL TABLES IN SCHEMA ai_chatbot TO PUBLIC;",
#     "GRANT USAGE, SELECT, UPDATE ON ALL SEQUENCES IN SCHEMA ai_chatbot TO PUBLIC;",
#     "ALTER DEFAULT PRIVILEGES IN SCHEMA ai_chatbot GRANT ALL ON TABLES TO PUBLIC;",
#     "ALTER DEFAULT PRIVILEGES IN SCHEMA ai_chatbot GRANT USAGE, SELECT, UPDATE ON SEQUENCES TO PUBLIC;",
# ]

# for grant in grants:
#     try:
#         client.execute(grant)
#     except Exception as e:
#         print(f"Warning on '{grant[:50]}...': {e}")

# print("✓ Permissions granted to PUBLIC for both schemas")
# client.close()